# CS639 Colab Runner

This notebook mounts Google Drive, installs dependencies, loads the default config, runs a small Qwen smoke test, and then executes the full experiment suite.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/CS639'
CONFIG_PATH = f'{PROJECT_ROOT}/configs/default.yaml'
MODEL_PATH = f'{PROJECT_ROOT}/models/Qwen3-4B-Instruct-2507'
DATASET_ROOT = f'{PROJECT_ROOT}/dataset'
OUTPUT_ROOT = f'{PROJECT_ROOT}/outputs'
RUNTIME_CONFIG_PATH = f'{PROJECT_ROOT}/configs/colab_runtime.yaml'
PILOT_CONFIG_PATH = f'{PROJECT_ROOT}/configs/colab_pilot.yaml'
TUNED_CONFIG_PATH = f'{PROJECT_ROOT}/configs/colab_runtime_tuned.yaml'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CONFIG_PATH   =', CONFIG_PATH)
print('MODEL_PATH    =', MODEL_PATH)
print('DATASET_ROOT  =', DATASET_ROOT)
print('OUTPUT_ROOT   =', OUTPUT_ROOT)
print('RUNTIME_CONFIG=', RUNTIME_CONFIG_PATH)
print('PILOT_CONFIG  =', PILOT_CONFIG_PATH)
print('TUNED_CONFIG  =', TUNED_CONFIG_PATH)

In [ ]:
%cd /content
!python -m pip install --upgrade pip
!python -m pip install -r "$PROJECT_ROOT/requirements.txt"

In [ ]:
%cd /content
!python "$PROJECT_ROOT/scripts/download_datasets.py" --root-dir "$DATASET_ROOT" --hf-cache-dir "$DATASET_ROOT/hf_cache" --overwrite

In [ ]:
import os
import sys
from pathlib import Path

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

required_paths = [
    PROJECT_ROOT,
    CONFIG_PATH,
    MODEL_PATH,
    DATASET_ROOT,
]

for path in required_paths:
    exists = Path(path).exists()
    print(f'{path}:', 'OK' if exists else 'MISSING')

In [ ]:
from src.config import load_config
import yaml

config = load_config(CONFIG_PATH)
config['model']['model_name_or_path'] = MODEL_PATH
config['datasets']['root_dir'] = DATASET_ROOT
config['experiment']['output_dir'] = OUTPUT_ROOT

with open(RUNTIME_CONFIG_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

pilot_config = load_config(RUNTIME_CONFIG_PATH)
pilot_config['datasets']['dev_split'] = {
    'gsm8k': 3,
    'bbh_date_understanding': 2,
    'bbh_logical_deduction_three_objects': 2,
    'mmlu': 2,
}
pilot_config['datasets']['eval_split'] = {
    'gsm8k': 10,
    'bbh_date_understanding': 5,
    'bbh_logical_deduction_three_objects': 5,
    'mmlu': 10,
}
pilot_config['reasoning']['tau_0'] = 0.67
pilot_config['reasoning']['tau_1'] = 0.5
pilot_config['reasoning']['tau_expand'] = 0.05
pilot_config['reasoning']['lambda_cost'] = 0.1

with open(PILOT_CONFIG_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(pilot_config, f, sort_keys=False)

print('Wrote runtime config to:', RUNTIME_CONFIG_PATH)
print('Wrote pilot config to:  ', PILOT_CONFIG_PATH)
config

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from src.llm.loader import build_llm

llm = build_llm(config['model'])
print('Loaded model from:', llm.model_name_or_path)
print('Runtime device:', llm.device)

In [ ]:
prompt = '''Solve the following question step by step, then provide a final answer.\n\nQuestion: If Alice has 12 apples and gives 5 away, how many apples does she have left?\nReasoning:'''

result = llm.generate(prompt, max_new_tokens=128, temperature=0.0, n=1)[0]
print(result['text'])
print('\nPrompt tokens:', result['prompt_tokens'])
print('Completion tokens:', result['completion_tokens'])
print('Latency (sec):', result['latency_sec'])

In [ ]:
from src.parsers.answers import extract_final_answer

print('Parsed final answer:', extract_final_answer(result['text']))

## Pilot Run

Once the smoke test works, first run the full pipeline on a small pilot config. If the outputs look reasonable, then run the full tuning and full experiment suite.

In [ ]:
%cd /content
!python "$PROJECT_ROOT/scripts/run_all_experiments.py" --config "$PILOT_CONFIG_PATH"

## Full Run

If the pilot run looks reasonable, tune the adaptive thresholds on the full frozen dev split, then run the full experiment suite on the tuned config.

In [ ]:
%cd /content
!python "$PROJECT_ROOT/scripts/tune_reasoning.py" --config "$RUNTIME_CONFIG_PATH" --output-config "$TUNED_CONFIG_PATH"

In [ ]:
%cd /content
!python "$PROJECT_ROOT/scripts/run_all_experiments.py" --config "$TUNED_CONFIG_PATH"